In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sacrebleu.metrics import BLEU, CHRF

In [ ]:
def calculate_exact_match(predictions, references):
    """Calculate exact match accuracy."""
    matches = sum([1 for pred, ref in zip(predictions, references) if pred.strip() == ref.strip()])
    return (matches / len(predictions)) * 100

def evaluate_and_print(lang_name, split_name, predictions, references):
    """Calculate and print all metrics for a given split."""
    bleu = BLEU()
    chrf = CHRF()
    
    bleu_score = bleu.corpus_score(predictions, [references]).score
    chrf_score = chrf.corpus_score(predictions, [references]).score
    exact_match = calculate_exact_match(predictions, references)
    
    print(f"{lang_name:10s} {split_name:5s} {bleu_score:10.6f} {chrf_score:10.6f} {exact_match:10.6f}")
    
    return {
        'bleu': bleu_score,
        'chrf': chrf_score,
        'exact_match': exact_match
    }

In [23]:
def train_hybrid(train_df, char_ngram=(2, 5), word_ngram=(1, 2), weight=0.7):
    """
    Train hybrid character + word n-gram model.
    
    Args:
        train_df: DataFrame with 'source' and 'target' columns
        char_ngram: Character n-gram range
        word_ngram: Word n-gram range  
        weight: Weight for character features (1-weight for word features)
    """
    # Character-level vectorizer
    char_vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=char_ngram,
        lowercase=True
    )
    
    # Word-level vectorizer
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=word_ngram,
        lowercase=True
    )
    
    train_sources = train_df["source"].tolist()
    
    # Fit both vectorizers
    X_char = char_vectorizer.fit_transform(train_sources)
    X_word = word_vectorizer.fit_transform(train_sources)
    
    # Combine features with weighting
    X_char_weighted = X_char * weight
    X_word_weighted = X_word * (1 - weight)
    X_train = hstack([X_char_weighted, X_word_weighted])
    
    train_targets = train_df["target"].tolist()
    
    return (char_vectorizer, word_vectorizer, weight), X_train, train_targets


In [24]:
def nn_hybrid(vectorizers, X_train, train_targets, test_sources):
    """
    Nearest neighbor retrieval with hybrid vectorizers.
    
    Args:
        vectorizers: Tuple of (char_vectorizer, word_vectorizer, weight)
        X_train: Training feature matrix
        train_targets: Training target translations
        test_sources: Test source sentences
    """
    char_vectorizer, word_vectorizer, weight = vectorizers
    
    # Transform test data with both vectorizers
    X_char = char_vectorizer.transform(test_sources)
    X_word = word_vectorizer.transform(test_sources)
    
    # Apply same weighting as training
    X_char_weighted = X_char * weight
    X_word_weighted = X_word * (1 - weight)
    X_test = hstack([X_char_weighted, X_word_weighted])
    
    # Find nearest neighbors
    sims = cosine_similarity(X_test, X_train)
    nn_indices = np.argmax(sims, axis=1)
    preds = [train_targets[i] for i in nn_indices]
    
    return preds

In [25]:
hing_train = pd.read_csv("data/hinglish_train.csv")
hing_val   = pd.read_csv("data/hinglish_val.csv")
hing_test  = pd.read_csv("data/hinglish_test.csv")

span_train = pd.read_csv("data/spanglish_train.csv")
span_val   = pd.read_csv("data/spanglish_val.csv")
span_test  = pd.read_csv("data/spanglish_test.csv")

print("Dataset sizes:")
print(f"Hinglish: train={len(hing_train)}, val={len(hing_val)}, test={len(hing_test)}")
print(f"Spanglish: train={len(span_train)}, val={len(span_val)}, test={len(span_test)}")


Dataset sizes:
Hinglish: train=743, val=93, test=93
Spanglish: train=844, val=105, test=106


In [26]:
print("HINGLISH BASELINE (Hybrid: char 2-5 + word 1-2, weight=0.7)")

hing_vectorizers, hing_X_train, hing_train_targets = train_hybrid(
    hing_train, 
    char_ngram=(2, 5), 
    word_ngram=(1, 2), 
    weight=0.7
)

hing_val_sources = hing_val["source"].tolist()
hing_val_refs = hing_val["target"].tolist()
hing_val_preds = nn_hybrid(hing_vectorizers, hing_X_train, hing_train_targets, hing_val_sources)

hing_test_sources = hing_test["source"].tolist()
hing_test_refs = hing_test["target"].tolist()
hing_test_preds = nn_hybrid(hing_vectorizers, hing_X_train, hing_train_targets, hing_test_sources)

print("\nFirst 10 Hinglish predictions:")
for i in range(10):
    print(f"Sample {i+1}")
    print("SOURCE :", hing_test_sources[i])
    print("PRED   :", hing_test_preds[i])
    print("REF    :", hing_test_refs[i])

HINGLISH BASELINE (Hybrid: char 2-5 + word 1-2, weight=0.7)

First 10 Hinglish predictions:
Sample 1
SOURCE : Beach town ka mayor kuch gadbad type ka aadmi hai kyunki woh beach goers ko beach ke khatron ke baare mein nahi batana chahta.
PRED   : yeah, he is a very selfish type of person I think in reality
REF    : The mayor of the beach town is kind of the bad guy as he doesn't want to tell the beach goers how dangerous the beach is.
Sample 2
SOURCE : Firstly, Rotten Tomatoes par ise great reviews mile ! ye ek former human ke bare me he jo apni wife ki death ka revenge lene ke liye criminal underworld me lotata he.
PRED   : This film has a great rotten tomatoes score
REF    : Firstly, it received great reviews on Rotten Tomatoes! And it's about a former human who returns to the criminal underworld to extract revenge following the death of his wife!
Sample 3
SOURCE : mujhe yeh bhi laga ki yeh bohot unfair tha jo bogo ne usse resignation maanga kyu wo abhi nayi thi force me
PRED   : I ag

In [27]:
print("SPANGLISH BASELINE (Hybrid: char 2-5 + word 1-2, weight=0.7)")

span_vectorizers, span_X_train, span_train_targets = train_hybrid(
    span_train, 
    char_ngram=(2, 5), 
    word_ngram=(1, 2), 
    weight=0.7
)

span_val_sources = span_val["source"].tolist()
span_val_refs = span_val["target"].tolist()
span_val_preds = nn_hybrid(span_vectorizers, span_X_train, span_train_targets, span_val_sources)

span_test_sources = span_test["source"].tolist()
span_test_refs = span_test["target"].tolist()
span_test_preds = nn_hybrid(span_vectorizers, span_X_train, span_train_targets, span_test_sources)

print("\nFirst 10 Spanglish predictions:")
for i in range(10):
    print(f"Sample {i+1}")
    print("SOURCE :", span_test_sources[i])
    print("PRED   :", span_test_preds[i])
    print("REF    :", span_test_refs[i])

SPANGLISH BASELINE (Hybrid: char 2-5 + word 1-2, weight=0.7)

First 10 Spanglish predictions:
Sample 1
SOURCE : RB: Bueno, me gusta pensar que representa quality, que, you know, if alguien se topa con una Virgin company, they -- CA: They are quality, Richard. Come on now, todo el mundo habla de quality --¿el spirit?
PRED   : But I would like to talk further about the combination of light and darkness as a quality in our life.
REF    : RB: Well, I like to think it stands for quality, that you know, if somebody comes across a Virgin company, they -- CA: They are quality, Richard. Come on now, everyone says quality. Spirit?
Sample 2
SOURCE : Presentamos más de 300 muestras de mushrooms que fueron hervidos in hot water, y el micelio harvesting estos extracellular metabolites.
PRED   : So the biggest moment for me, though, my most important job now is I am a dad myself, and I have two beautiful daughters, and my goal is to surround them by inspiration, by the books that are in every single 

In [28]:
print("EVALUATION METRICS")
print(f"{'lang':>10s} {'split':>5s} {'BLEU':>12s} {'chrF':>12s} {'ExactMatch':>12s} \\")

# Evaluate all splits
evaluate_and_print("hinglish", "val", hing_val_preds, hing_val_refs)
evaluate_and_print("hinglish", "test", hing_test_preds, hing_test_refs)
evaluate_and_print("spanglish", "val", span_val_preds, span_val_refs)
evaluate_and_print("spanglish", "test", span_test_preds, span_test_refs)


EVALUATION METRICS
      lang split         BLEU         chrF   ExactMatch \
hinglish   val    26.796699  37.869461  22.580645
hinglish   test   27.526264  38.125039  16.129032
spanglish  val     1.543735  22.532027   0.000000
spanglish  test    1.535440  22.701999   0.000000


{'bleu': 1.5354401627326824, 'chrf': 22.701998865902578, 'exact_match': 0.0}

In [ ]:
hing_test_copy = hing_test.copy()
hing_test_copy["baseline_pred"] = hing_test_preds
hing_test_copy.to_csv("hinglish_hybrid_baseline_predictions.csv", index=False)

span_test_copy = span_test.copy()
span_test_copy["baseline_pred"] = span_test_preds
span_test_copy.to_csv("spanglish_hybrid_baseline_predictions.csv", index=False)

print("\nPredictions saved to CSV files.")